# Tissue-Level Expression: GTEx


**Estimated time:** 25 minutes

Compare median expression of the 25 genes in two GTEx heart tissues and identify
which tissue has the higher reference value for selected genes.


## Tissue-level expression

The source paper gives us 54 variants in 25 genes, but its candidate genetic variant table does
not show where those genes are expressed. Our next question is how median gene
expression compares between two reference heart tissues.


### What GTEx contains and why we use it

GTEx stands for **Genotype-Tissue Expression**. This resource
contains gene-expression data from human tissue donors. It provides a reference
for comparing expression across tissues rather than a cohort selected for heart
failure. This lesson uses the two heart tissues represented in GTEx: atrial
appendage and left ventricle.

Expression is reported as median transcripts per million, or TPM. TPM
summarizes gene-expression abundance while accounting for transcript length and
sequencing depth. The median gives one value across the GTEx samples represented
for a tissue.

Learn more about the resource and its endpoints in the
[GTEx Portal API documentation](https://gtexportal.org/api/v2/docs).


### How to read GTEx values

Use the median TPM values as a descriptive comparison between reference tissues.
GTEx uses bulk tissue, so each value combines signals from the cell types present
in those samples.

**Why GTEx helps**
The candidate genetic variant table identifies genes but does not provide tissue expression.
GTEx adds a reference value for each gene in atrial appendage and left ventricle.
A higher median in one tissue can suggest which heart tissue to examine in
follow-up work.


## Querying the GTEx API

The live request first matches the paper's 25 gene symbols to GENCODE v39 IDs.
A second request asks for GTEx v10 median expression in two heart tissues. This
module specifies v10 so the live request and dated teaching response refer to
the same GTEx release.


### Prepare the gene list and API helper

Load the published variants, select the 25 unique gene symbols, and import the
GTEx wrapper.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Locate the repository root.
REPO_ROOT = Path.cwd() if Path("api_helpers.py").exists() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Import the API wrapper.
from api_helpers import fetch_gtex_context

# Load the published variants.
DATA_DIR = REPO_ROOT / "data"
variants = pd.read_csv(DATA_DIR / "variants.csv")
gene_symbols = sorted(variants["gene_symbol"].unique())
pd.Series(gene_symbols, name="gene_symbol").head()


`gene_symbols` contains the 25 unique gene symbols in alphabetical order.
`variants` retains all 54 published rows.


### Request heart-tissue expression

Request median expression for the 25 genes in atrial appendage and left
ventricle. In the returned table, look for the gene and tissue identifiers,
the median TPM value, and the retrieval date.


In [ ]:
# Query the 25 genes.
gtex = fetch_gtex_context(gene_symbols)

# If GTEx is unavailable, comment out the line above and uncomment this backup.
# gtex = pd.read_csv(DATA_DIR / "gtex_expression.csv").assign(unit="TPM")

# Inspect the returned fields.
gtex.head()


Each row contains one gene-tissue combination. `median_tpm` is the median
transcript abundance reported for that tissue. `gencode_id`, `dataset_id`, and
`retrieved_date` record the identifiers and retrieval context. The `unit` field
confirms that the returned expression values are TPM.


### Check the returned data

Summarize the response before comparing tissues. For this request, expect 25
genes, 50 gene-tissue rows, two tissues, and TPM units.


In [ ]:
response_summary = pd.Series(
    {
        "genes returned": gtex["gene_symbol"].nunique(),
        "tissue rows returned": len(gtex),
        "tissues returned": gtex["tissue_name"].nunique(),
        "units returned": ", ".join(sorted(gtex["unit"].dropna().unique())),
    },
    name="value",
).to_frame()
response_summary


The summary makes missing genes or tissue rows visible before the expression
values are reshaped.

**Live data**
Values are requested directly from the GTEx v10 API. API availability can
change, so the dated v10 teaching response provides a backup.


### Request sequence

**GTEx API request details**
The `fetch_gtex_context()` wrapper resolves gene symbols, requests expression,
and returns a tidy table.

| Step | Endpoint | Important parameters |
|---|---|---|
| Resolve symbols | `/api/v2/reference/gene` | Gene symbols, GENCODE v39, and GRCh38 |
| Request expression | `/api/v2/expression/medianGeneExpression` | GENCODE IDs, GTEx v10, atrial appendage, and left ventricle |

The complete implementation and endpoint constants are maintained in
[`api_helpers.py`](../api_helpers.py).


## Compare heart tissues

Reshape the API result into a gene-by-tissue table, add the number of published
rows per gene, and then plot a readable subset.


### Summarize genes and tissues

Reshape the GTEx values so each gene has one column per tissue. Then add the
number of published rows connected to each gene for reference.


In [ ]:
# Count published rows per gene.
variant_counts = variants.groupby("gene_symbol").size().rename("variant_rows")

# Reshape tissues into columns.
heart_expression = (
    gtex.pivot(
        index="gene_symbol",
        columns="tissue_name",
        values="median_tpm",
    )
    .join(variant_counts)
    .sort_values("Heart - Left Ventricle", ascending=False)
)
heart_expression


Each row now represents one gene. The two tissue columns contain median TPM,
and `variant_rows` counts its occurrences in the published variant table. In
the dated teaching data, *DES* and *MYH7* have the highest left-ventricle median
TPM values. The `variant_rows` column describes the source table, while the two
TPM columns contain the GTEx expression context.


### Plot genes with the highest expression

Plot the ten genes with the highest median left-ventricle expression. This
smaller view keeps labels readable; the complete 25-gene table remains above.


In [ ]:
# Select the top ten left-ventricle values.
top_heart_genes = heart_expression.nlargest(
    10,
    "Heart - Left Ventricle",
)

# Plot both heart tissues.
axis = top_heart_genes.loc[
    :, ["Heart - Atrial Appendage", "Heart - Left Ventricle"]
].plot.bar(
    color=["#3d64b3", "#764c82"],
    figsize=(10, 5),
)
axis.set_ylabel("Median expression (TPM)")
axis.set_xlabel("Gene from the source paper")
axis.set_title("GTEx v10 heart-tissue expression")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


*DES* and *MYH7* have higher median expression in left ventricle than atrial
appendage, while *ACTC1* is higher in atrial appendage. These are bulk-tissue
patterns that motivate the cell-type question in the HuBMAP lesson.


### Compare the atrial ranking

Rank the same genes by atrial-appendage expression and compare the result with
`top_heart_genes`.


```python
# Rank genes by atrial-appendage expression.
top_atrial_genes = heart_expression.nlargest(
    10,
    "______",
)
top_atrial_genes.head()
```

**Hint**
Use the `Heart - Atrial Appendage` column.

**Solution**


In [ ]:
# Rank genes by atrial-appendage expression.
top_atrial_genes = heart_expression.nlargest(
    10,
    "Heart - Atrial Appendage",
)
top_atrial_genes.head()


*ACTC1* ranks first in atrial appendage, while *DES* ranks first in left
ventricle. *LMNA* enters the atrial top 10, and *MYL3* appears only in the
left-ventricle top
10. The selected tissue changes which genes receive the most prominent
expression context.


## Check your understanding


What does `median_tpm` represent in this lesson?

- Median gene-level TPM across the GTEx samples represented for a tissue

  > Correct. Each value summarizes gene-level expression across the GTEx
  > samples represented for one tissue.

- Expression measured in the heart-failure study participants

  > The values come from the GTEx reference resource rather than the
  > heart-failure study cohort.

- The number of published variant rows for a gene

  > The separate `variant_rows` column records how often a gene occurs in the
  > published candidate genetic variant table.



What does GTEx add to the published variant list?

- Tissue-level expression context

  > Correct. GTEx describes gene expression in the selected heart tissues.

- A new ACMG classification

  > The study classification remains attached to the published variant rows.


## Key points

- The GTEx v10 response contains median TPM for all 25 genes in two bulk heart
  tissues.
- *DES* and *MYH7* have the highest left-ventricle median TPM in the dated data.
- Ranking by atrial appendage instead of left ventricle changes the top-10 gene
  set.

**Next:** GTEx summarizes whole heart tissues, which contain several cell types.
Use HuBMAP to ask whether the same genes have indexed expression in ventricular
cardiac myocytes.
